# 07 · Bounds and bases — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1. WLS, NNLS, and fixed rake

NNLS removes negative amplitudes; fixed rake additionally reduces dimension.
Compare fit loss and active zeros, not only the smoother-looking map.

## 2–3. Upper and component bounds

The bias threshold is where many parameters hit the cap and residual metrics
rise. Component bounds must follow the solved basis.

## 4. Reduction matrix

For rake 90°, the strike block is zero and dip block is identity. Matrix
predictions therefore match the one-component solve exactly.


In [ ]:
kwargs = dict(
    datasets=gnss, components="dip", regularization="laplacian",
    regularization_strength=0.2,
)
free = geodef.solve(fault, **kwargs)
nnls = geodef.solve(fault, bounds=(0.0, None), **kwargs)
rake = geodef.solve(
    fault, datasets=gnss, components="rake", rake=90.0,
    regularization="laplacian", regularization_strength=0.2,
    bounds=(0.0, None),
)
G_full = geodef.greens.matrix(fault, gnss)
reduction = np.vstack([np.zeros((fault.n_patches, fault.n_patches)), np.eye(fault.n_patches)])
assert np.allclose(G_full @ reduction @ rake.slip_vector, rake.predicted)
print("minimum free/NNLS:", free.dip_slip.min(), nnls.dip_slip.min())
print("reduced chi2 free/NNLS/rake:", free.reduced_chi2, nnls.reduced_chi2, rake.reduced_chi2)


## Interpretation and alternatives

A basis can be more consequential than a bound because it excludes an entire orthogonal component everywhere.
